# Unit 01｜主动学习闭环与标签揭示线

## Goal

用固定预测表完成一次 UCB query 和同预算 Random query，区分选样与标签返回。

本 Notebook 是确定性的人工教学实验，不是学习者已完成的研究，
也不是下游任务实验结果。


## Setup

候选特征和模型预测在 query 前可见；`oracle_values` 只在候选编号固定后读取。


In [1]:
import numpy as np
import pandas as pd

candidate_ids = np.array([f"C{i:02d}" for i in range(12)])
X_pool = np.column_stack([
    np.linspace(0.10, 0.90, 12),
    np.linspace(80.0, 140.0, 12),
])
prediction_mean = np.array(
    [5.1, 5.4, 5.8, 6.0, 6.1, 6.2, 6.3, 6.5, 6.6, 6.7, 6.8, 6.9]
)
prediction_std = np.array(
    [0.3, 0.5, 0.2, 0.9, 0.4, 0.2, 1.1, 0.3, 0.7, 0.2, 0.4, 0.6]
)

# 教学 Oracle：query 前不能用它调整策略。
oracle_values = np.array(
    [4.9, 5.7, 5.6, 6.8, 6.0, 6.1, 7.5, 6.4, 7.0, 6.6, 6.9, 7.2]
)

print("X_pool shape:", X_pool.shape)
print("候选数:", len(candidate_ids))


X_pool shape: (12, 2)
候选数: 12


## Steps

按顺序执行。每个变量第一次出现时，先确认它的类型、形状和标签权限。


### 1. 只根据预测计算 UCB

`beta` 在查看任何候选真实标签前固定。


In [2]:
beta = 1.0
ucb_score = prediction_mean + beta * prediction_std
query_position = int(np.argmax(ucb_score))
query_id = candidate_ids[query_position]

ranking = pd.DataFrame({
    "candidate_id": candidate_ids,
    "prediction_mean": prediction_mean,
    "prediction_std": prediction_std,
    "ucb_score": ucb_score,
}).sort_values(
    ["ucb_score", "candidate_id"],
    ascending=[False, True],
)
print(ranking.head(5).to_string(index=False))
print("UCB query:", query_id)


candidate_id  prediction_mean  prediction_std  ucb_score
         C11              6.9             0.6        7.5
         C06              6.3             1.1        7.4
         C08              6.6             0.7        7.3
         C10              6.8             0.4        7.2
         C03              6.0             0.9        6.9
UCB query: C11


### 2. 固定同预算 Random query

Random 也在标签揭示前确定，且只选择一个候选。


In [3]:
random_rng = np.random.default_rng(2026)
random_position = int(random_rng.integers(len(candidate_ids)))
random_query_id = candidate_ids[random_position]
print("Random query:", random_query_id)


Random query: C10


### 3. 越过标签揭示线

现在 query 已经固定，才模拟实验返回真实性能。


In [4]:
selected_y = float(oracle_values[query_position])
random_selected_y = float(oracle_values[random_position])

query_summary = pd.DataFrame([
    {
        "strategy": "ucb",
        "candidate_id": query_id,
        "observed_y": selected_y,
    },
    {
        "strategy": "random",
        "candidate_id": random_query_id,
        "observed_y": random_selected_y,
    },
])
print(query_summary.to_string(index=False))


strategy candidate_id  observed_y
     ucb          C11         7.2
  random          C10         6.9


## Checks

这些断言检查形状、预算和无重复等机械条件；通过断言不代表研究结论已经成立。


In [5]:
assert X_pool.shape == (12, 2)
assert query_id in candidate_ids
assert random_query_id in candidate_ids
assert len(query_summary) == 2
assert query_summary["observed_y"].notna().all()
print("Unit 01 checks passed.")


Unit 01 checks passed.


## Next Steps

完成练习并用自己的话画出五模块闭环，然后进入 Unit 02。不要根据这一轮胜负评价策略优劣。
